In [0]:
import datetime

today = datetime.date.today() 
month = today.month
year = today.year

print(month, year)

In [0]:
gold_path = f's3://my-spotify-delta-lakehouse/gold/played_in_{month}_{year}/'

In [0]:
    %sql
    SHOW CATALOGs;

In [0]:
source_table = spark.sql(f"""
SELECT
p.played_at,
p.time_of_day as time_of_day,
p.track_id,
p.track_name,
t.popularity as track_popularity,
p.first_artist_id,
a.name AS artist_name,
a.genre_1 AS artist_genre_1,
a.genre_2 AS artist_genre_2,
a.popularity AS artist_popularity,
p.album_id,
b.name AS album_name,
b.popularity AS album_popularity,
DATE(b.release_date) as album_release_date
FROM my_spotify.silver.user_recent_played p
JOIN my_spotify.silver.artists a
ON p.first_artist_id = a.id
JOIN my_spotify.silver.albums b
ON p.album_id = b.id
JOIN my_spotify.silver.tracks t
ON p.track_id = t.id
WHERE MONTH(PLAYED_AT) = {month}
AND YEAR(PLAYED_AT) = {year}""")

In [0]:
from delta.tables import DeltaTable

try:
    target = DeltaTable.forPath(spark, gold_path)
    source_table.alias("source").merge(
        target.alias("target"),
        "source.played_at = target.played_at"
    ).whenNotMatchedInsertAll().execute()
except Exception:
    source_table.write.format("delta").save(gold_path)
    spark.sql(f"CREATE TABLE my_spotify.gold.{month}_{year} USING DELTA LOCATION '{gold_path}'")

In [0]:
%skip
%sql
ALTER TABLE my_spotify.gold.12_2025
RENAME TO  my_spotify.gold.played_in_12_2025;